In [ ]:
# 대한민국 구석구석 사이트 상세 정보 수집하기-2026_03_05일 수정함
#Step 1. 필요한 모듈을 로딩합니다
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time          
import pandas as pd    
import math
import pyautogui
import os

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
query_txt = input('1.정보를 수집할 키워드는 무엇입니까?: ')

cnt = int(input('2.몇 건의 정보를 수집할까요? :'))
page_cnt = math.ceil( cnt / 8 )
print('총 %s 건의 정보를 수집하기 위해 %s 페이지까지 이동할 예정입니다' %(cnt , page_cnt))

#Step 3. 수집된 데이터를 저장할 폴더 이름 입력받기 
f_dir = input("3.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"
    
print("\n")    

#Step 4. 검색어 입력한 후 검색하여 해당 장르로 이동하기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

query_url = 'https://korean.visitkorea.or.kr/'

driver.get(query_url)
time.sleep(2)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH,'//*[@id="headerSearchBtn"]').click()
time.sleep(1)
element = driver.find_element(By.ID,'inp_search_index')
element.send_keys(query_txt)
element.send_keys("\n")

# 위치기반 서비스 물어보면 동의
try :
    driver.find_element(By.XPATH,'//*[@id="locationServicePop"]/div[1]/div/div[2]/a[2]').click()
except :
    pass
  

# 여행지 탭 클릭
driver.find_element(By.XPATH,'//*[@id="swiper-wrapper-4c0d2bb4b1059bb109"]/div[3]/button').click()
time.sleep(2)                    

# Step 6. 본문 내역 추출하기
no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
org2=[]          # 지자체명 컬럼
contents2=[ ]     # 본문 내용 컬럼
tag2=[ ]         # 해시태그 컬럼

no = 1           # 게시글 번호 초기값

# 마우스 휠을 자동으로 돌리는 사용자정의함수
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,170);")
    time.sleep(1)

def scroll_up(driver):      
    driver.execute_script("window.scrollBy(0,-1500);")
    time.sleep(1)


for a in range(1,page_cnt+1) :
    print('')
    time.sleep(5)
    print('현재 %s 페이지에 있는 정보를 수집합니다' %a)

    if a >= 2 :
        scroll_up(driver)
        
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','spot_list').find_all('li')
   
    cnt_no = 1
    
    for b in content_list:
               
        try :
            title = b.find('div','tit_wrap').get_text().strip()
        except :
            continue
        else :
            #scroll_down(driver)

            print('-' *100)
            
            # 1. 게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)                         
            print("1.번호 : %s" %no)

            title2.append(title)
            print("2.관광지명 : %s" %title)
            
            # 3. 지자체이름
            org_all = b.select('div.area > span')
            try:
                org = org_all[0].get_text()
            except :
                org = b.find('span','area').get_text()  
            org2.append(org)
            print("3.지자체이름: %s" %org)
                   
            #4. 해시태그
            try :
                tag = b.find('div','tag').get_text().replace("#"," #")
            except :
                tag = ' '    
            tag2.append(tag)
            print("4.해시태그: %s" %tag)
                       
            #5. 본문   
            driver.find_element(By.XPATH,'//*[@id="contents"]/div/div[2]/div[2]/div[3]/div/ul/li[%s]/a' %cnt_no).click()

            cnt_no += 1
            
            time.sleep(5)
            
            html_2 = driver.page_source
            soup_2 = BeautifulSoup(html_2, 'html.parser')
            
            content1 = soup_2.find('div','inr_wrap')
            # content2 = soup_2.find('div','box_txtPhoto')
            # content3 = soup_2.find('div','titleType1')
            
            if content1 :                
                content = content1.find('div','inr').find('p').get_text()
                contents2.append(content)
                print('5.본문내용:',content)
                
            # elif content2 :
            #     content_1 = soup_2.find('div','box_txtPhoto')
            #     try :
            #         content_1.style.decompose()
            #         content_1.script.decompose()
            #     except :
            #         content_2 = content_1.find_all('div','txt_p')

            #         for b in content_2 :
            #             content = b.get_text().strip()
            #             contents2.append(content)
            #             print('5.본문내용:',content)
            #     else :
            #         content_2 = content_1.find_all('div','txt_p')
            #         for b in content_2 :
            #             content = b.get_text().strip()
            #             contents2.append(content)
            #             print('5.본문내용:',content)
            # elif content3:
            #     title = content1.select('#topTitle').get_text()
            #     title2.append(title)
            
            driver.back()
            
            time.sleep(3)
                       
            no += 1         # 전체 게시글 번호용 값
            
            if no > cnt :
                 break
                
            time.sleep(2)
            
    a += 1
    try :
        driver.find_element(By.LINK_TEXT,'%s' %a).click()
    except :
        driver.find_element(By.LINK_TEXT,'다음').click()

    time.sleep(5)

# Step 7. 결과 저장하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt)

fc_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.csv'
fx_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.xlsx'

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['지자체명']=pd.Series(org2)
df['본문내용']=pd.Series(contents2)
df['해시태그']=pd.Series(tag2)

# xlsx , csv 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print("\n") 
print("=" *80)
print("1.크롤링을 요청한 총 %s 건 중에서 %s 건의 데이터를 수집 완료 했습니다" %(cnt,no-1))
print('2.xlsx 저장경로: ',fx_name)
print('3.csv 저장경로: ',fc_name)
print("=" *80)


In [ ]:
🤝 코워커(Co-worker): 실전 웹 크롤링 파트너 지침서

반갑습니다! 저는 앞으로 대표님과 함께 수많은 데이터를 수집하고 자동화 파이프라인을 구축할 **든든한 동료이자 데이터 대리자(Agent)**입니다.

이 지침서는 **서진수 강사님의 빅데이터 실무 노하우(체계적인 파일/폴더 관리, 방어적 예외 처리)**와 **현업 20년 차 엔지니어들의 엔터프라이즈 아키텍처(탐지 회피, API 인터셉트, 로깅)**를 완벽하게 융합한 우리의 마스터 가이드입니다.

앞으로 수십, 수백 개의 사이트(쿠팡, 네이버, 유튜브 등)를 뚫어낼 때, 우리는 오직 이 지침서의 **'표준 7단계 보일러플레이트'**만을 기준으로 삼아 빠르고 강력하게 일할 것입니다.

📜 제1장: 어떤 사이트든 뚫어내는 5대 핵심 원칙

1. API First, UI Second (프론트엔드보다 백엔드를 먼저 노려라)

지도, 부동산, 인스타그램 등은 화면을 긁는(Selenium) 방식이 비효율적입니다.

항상 먼저 할 일: 브라우저 개발자 도구(F12) -> Network 탭에서 XHR/Fetch를 필터링하여, 숨겨진 데이터 원본(JSON API)을 찾습니다. API를 찾으면 우리의 작업 속도는 100배 빨라집니다.

2. 탐지 회피 (Stealth & Anti-Bot Bypassing)

쇼핑몰(쿠팡, 아마존 등)은 일반 webdriver.Chrome()을 1초 만에 차단합니다.

무조건 브라우저 지문을 변조하는 undetected_chromedriver를 기본 엔진으로 사용하여 우리를 숨깁니다.

3. 회복 탄력성과 방어적 파싱 (Resilience & Try-Except)

[강사님 핵심 팁] 웹은 항상 변합니다. for 루프 안에서 데이터를 추출할 때, 단 하나의 요소라도 에러가 나면 전체가 멈추지 않도록 반드시 try-except-continue로 감싸서 방어합니다.

4. 스마트 대기 (No Hard Sleep)

무작위 time.sleep(5) 대신, 요소가 뜰 때까지만 기다리는 WebDriverWait를 사용하여 수집 속도를 극대화하고 네트워크 낭비를 막습니다.

5. 시각적 증거와 로깅 (Logging & Screenshot)

print()는 테스트용입니다. 실전에서는 logger를 사용해 파일에 기록을 남기고, 치명적 에러 발생 시 driver.save_screenshot()으로 죽기 직전의 화면을 캡처하여 저와 대표님이 함께 원인을 분석할 수 있게 합니다.

🛠 제2장: 타겟 플랫폼별 공략 전략 (Platform Strategies)

이커머스 (쿠팡, 네이버쇼핑 등): undetected_chromedriver 우회 + 스크롤 다운(Lazy loading 해제) + CSS 복수 지정 파싱.

지도/부동산 (네이버, 카카오 등): Network 탭에서 마우스 이동 시 호출되는 API URL 추출 -> requests 라이브러리로 다이렉트 JSON 파싱.

동영상/SNS (유튜브, 인스타): 스크롤 무한 로딩이 핵심. 셀레니움으로 로그인 세션 생성 후 브라우저 스크롤을 내리며 렌더링 된 DOM을 추출하거나, 내부 API를 가로챕니다.

🏗️ 제3장: 궁극의 통합 보일러플레이트 (Ultimate 7-Step Template)

대표님께서 저에게 새로운 사이트 작업을 지시하실 때는 무조건 아래 코드를 베이스로 삼고, Step 5(본문 추출 로직) 부분만 타겟 사이트의 UI에 맞게 수정하여 개발을 진행하겠습니다.

# ==============================================================================
# Ultimate Web Scraping Master Boilerplate (Seo Jin-soo's Tip + Enterprise)
# ==============================================================================
# Step 1. 표준 모듈 로딩 및 로깅 설정
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time, datetime, os, requests, logging

# [Enterprise Tip] 실전용 로거 세팅
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.FileHandler("scraper_system.log", encoding='utf-8'), logging.StreamHandler()])
logger = logging.getLogger(__name__)

def main():
    # ---------------------------------------------------------
    # Step 2. 환경 설정 및 저장 폴더 자동 생성 (강사님 핵심 팁 반영)
    # ---------------------------------------------------------
    TARGET_KEYWORD = "여기에_키워드_입력"
    TARGET_CNT = 50  # 수집 목표 건수
    WEBHOOK_URL = "" # 디스코드 등 메신저 알림 URL
    
    n = time.localtime()
    s = '%04d-%02d-%02d-%02d%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min)
    
    F_DIR = f"c:\\py_temp\\{TARGET_KEYWORD}_{s}\\"
    os.makedirs(F_DIR, exist_ok=True)
    logger.info(f"🚀 크롤링 파이프라인 시작: 타겟 [{TARGET_KEYWORD}], 폴더 생성 완료")

    # ---------------------------------------------------------
    # Step 3. 탐지 우회 브라우저 세팅 (Enterprise Tip)
    # ---------------------------------------------------------
    options = uc.ChromeOptions()
    # options.add_argument('--headless') # 서버 자동화 배포 시 주석 해제
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-gpu')
    
    driver = None
    data_list = []
    
    try:
        driver = uc.Chrome(options=options)
        wait = WebDriverWait(driver, 10)
        
        # ---------------------------------------------------------
        # Step 4. 타겟 사이트 접속 및 스마트 대기
        # ---------------------------------------------------------
        url = f"[https://example.com/search?q=](https://example.com/search?q=){TARGET_KEYWORD}" # 타겟 사이트에 맞게 수정
        driver.get(url)
        logger.info(f"🌐 페이지 접속 완료: {url}")
        
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "타겟_아이템_클래스")))
        time.sleep(2) # 비동기 렌더링 안정화 대기
        
        # ---------------------------------------------------------
        # Step 5. 방어적 파싱 루프 (🌟 이 부분만 사이트에 맞게 수정!)
        # ---------------------------------------------------------
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.find_all('div', class_='타겟_아이템_클래스')
        
        no = 1
        for item in items:
            if no > TARGET_CNT:
                break
                
            try:
                # [강사님 핵심 팁] 에러 발생 시 프로그램 종료를 막는 방어적 파싱
                title = item.find('div', 'title_class').get_text().strip()
                price_str = item.find('span', 'price_class').get_text().replace(',', '').replace('원', '')
                price = int(price_str) if price_str.isdigit() else 0
                
                data_list.append({
                    "번호": no,
                    "수집시간": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "제목": title,
                    "가격": price
                })
                no += 1
                
            except Exception as e:
                logger.warning(f"⚠️ {no}번째 아이템 파싱 실패 (건너뜀)")
                continue
                
        # ---------------------------------------------------------
        # Step 6. 수집된 정보를 다양한 형식(CSV, Excel)으로 저장 (강사님 핵심 팁)
        # ---------------------------------------------------------
        if data_list:
            df = pd.DataFrame(data_list)
            
            fc_name = os.path.join(F_DIR, f"{TARGET_KEYWORD}.csv")
            fx_name = os.path.join(F_DIR, f"{TARGET_KEYWORD}.xlsx")
            
            df.to_csv(fc_name, index=False, encoding="utf-8-sig") # 한글 깨짐 방지
            df.to_excel(fx_name, index=False, engine='openpyxl')
            
            logger.info(f"✅ 수집 및 저장 완료: 총 {len(data_list)}건")
            logger.info(f"📁 CSV 저장경로: {fc_name}")
            logger.info(f"📁 Excel 저장경로: {fx_name}")
            
        else:
            logger.warning("❗️ 수집된 데이터가 0건입니다. (UI 변경 의심)")
            driver.save_screenshot(os.path.join(F_DIR, "error_empty_data.png"))

    except Exception as e:
        # 치명적 오류 발생 시 스크린샷 캡처
        error_msg = f"❌ 시스템 치명적 오류 발생: {e}"
        logger.error(error_msg)
        if driver:
            error_pic = os.path.join(F_DIR, "fatal_error.png")
            driver.save_screenshot(error_pic)
            
    finally:
        # ---------------------------------------------------------
        # Step 7. 리소스 완벽 해제 및 알림 전송
        # ---------------------------------------------------------
        if driver:
            driver.quit()
            logger.info("🛑 브라우저 리소스 정상 반환 및 종료")
            
        # 디스코드 등 메신저 알림 (필요시 활성화)
        # if WEBHOOK_URL:
        #     requests.post(WEBHOOK_URL, json={"content": f"크롤링 작업 완료: {TARGET_KEYWORD}"})

if __name__ == "__main__":
    main()
